![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial)

# Persistent Memory for AI Agents with Mem0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-memory-with-mem0/mem0_tutorial.ipynb)

## 🎯 Introduction

AI agents suffer from a fundamental limitation: **memory amnesia**. They forget everything after each conversation, can't learn from past interactions, and lose valuable context that could make them truly intelligent assistants.

While basic memory solutions store static information, **[Mem0](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fgithub%2Fnir&text=Mem0)** introduces a revolutionary approach: **self-improving memory** that automatically extracts insights and evolves with each interaction. Instead of just storing conversations, Mem0 builds an intelligent knowledge system that learns user preferences and provides contextual understanding.

## 🛠️ What We'll Build

We'll create an intelligent Personal AI Research Assistant that demonstrates Mem0's full capabilities:

It will:

*   Keep track of your preferences for depth, style, and formatting
*   Store research notes, summaries, and insights in vector memory for semantic recall
*   Pull out the entities in your notes (papers, authors, models) and use them to boost recall of related memories
*   Recognize when new questions relate to earlier discussions and surface the right context
*   Combine semantic similarity, keyword matching and entity matching to generate more informed answers over time

Instead of restarting from zero every session, the assistant continually builds on what it already knows - helping you form a growing, interconnected understanding of the topics you research.

## 🏗️ Memory Architecture Overview

**Mem0** is a memory layer for AI agents that automatically extracts important information from conversations, stores it in a vector database together with an index of the entities it mentions, and retrieves it later to provide long-term context and continuity.

Below is the high-level flow illustrating how Mem0 processes and stores memory.

![mem0_architecture](Assets/mem0_architecture.png)

*This diagram shows the mem0 1.x pipeline. mem0 2.x, which this notebook uses, keeps the extraction phase but writes new facts ADD-only (there is no UPDATE/DELETE pass any more), and it replaces the external graph database with entity matching inside the vector store.*

## 🎢 Learning Journey

**Phase 1: Vector Memory Foundation**

*   Basic Assistant: Build a working research assistant with vector memory
*   Memory Operations: See knowledge extraction in action, and what happens when a preference changes
*   Semantic Search: Experience intelligent retrieval beyond keywords
*   Self-Improvement: Watch memory evolve with each interaction

**Phase 2: Entity-Aware Retrieval**

*   Entity Extraction: See which papers, authors and models Mem0 pulls out of a message
*   Score Breakdown: See how semantic, keyword (BM25) and entity signals add up for each search hit
*   Limits: What entity matching does not do compared with a graph database

## ⚙️ Prerequisites and Environment Setup

For this tutorial, you'll need:

1. [OpenAI API Key](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fsignup&text=OpenAI%20API%20Key) - used for LLM processing (GPT-5.4 mini) and embeddings
2. *Optional:* a [Qdrant Cloud Account](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=qdrant&target=https%3A%2F%2Fcloud.qdrant.io&text=Qdrant%20Cloud%20Account) or any other vector store supported by Mem0 for vector memory. If you skip it, the notebook keeps its vectors in a local Qdrant folder, so the OpenAI key is all you need.

> **mem0 version note:** this notebook targets mem0 2.x (`mem0ai>=2.2`). mem0 2.0 changed the open-source API in two ways that matter here: `search()` and `get_all()` take the user as `filters={"user_id": ...}` and raise `ValueError` if you pass `user_id=` directly, and the external graph store (Neo4j and others) was removed from the open-source package. Earlier versions of this tutorial used Neo4j for graph memory; that version needs `mem0ai<2` (the last 1.x release is 1.0.11).

### Installation

Let's start by installing all required dependencies:

In [ ]:
# [nlp] adds spaCy for entity extraction and keyword lemmatization (needs Python 3.10-3.12);
# fastembed turns on BM25 keyword search in Qdrant. Without them Mem0 falls back to semantic-only search.
%pip install -q "mem0ai[nlp]>=2.2,<3" fastembed openai python-dotenv

import sys
!{sys.executable} -m spacy download en_core_web_sm

### Import Required Libraries

In [ ]:
# Core libraries
import os
import getpass

# Mem0 and OpenAI
from mem0 import Memory
from openai import OpenAI

### Required API Keys

Let's configure your API keys:

In [ ]:
def setup_api_keys():

    # OpenAI API Key (required)
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

    # Qdrant Cloud (optional): leave the URL empty to keep the vectors in a local Qdrant folder instead
    if "QDRANT_URL" not in os.environ:
        os.environ["QDRANT_URL"] = input("\nEnter your Qdrant Cloud URL (or press Enter to use a local Qdrant): ").strip()
    if os.environ["QDRANT_URL"] and "QDRANT_API_KEY" not in os.environ:
        os.environ["QDRANT_API_KEY"] = getpass.getpass("Enter your Qdrant API key: ")

    return bool(os.environ["QDRANT_URL"])

# Run the setup
use_qdrant_cloud = setup_api_keys()
print("Vector store:", "Qdrant Cloud" if use_qdrant_cloud else "local Qdrant folder (./qdrant_data)")

## Memory Configuration
## Phase 1: Vector Memory

Now let's configure Mem0's memory system. We're starting with **vector-based memory** - the foundation that makes everything else possible.


### Understanding the Configuration

Each component serves a specific purpose in the memory pipeline:

- **🤖 LLM Provider**: Extracts the facts worth remembering from each message
- **📊 Embedder**: Converts text into semantic vectors
- **🗄️ Vector Store**: Stores embeddings for similarity search

In [ ]:
LLM_MODEL = "gpt-5.4-mini"  # used by Mem0 and by the assistant below

if use_qdrant_cloud:
    qdrant_connection = {"url": os.environ["QDRANT_URL"], "api_key": os.environ["QDRANT_API_KEY"]}
else:
    qdrant_connection = {"path": "qdrant_data"}  # embedded Qdrant, saved next to this notebook (delete the folder to start fresh)

config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": LLM_MODEL,
            "temperature": 0.1,
            "max_tokens": 2000,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-large",
            "embedding_dims": 3072,
        }
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            **qdrant_connection,
            "collection_name": "research_assistant_vectors",
            "embedding_model_dims": 3072,  # Must match embedding model
        }
    },
}

print("Vector Memory Configuration Created!")

### Initialize Memory System

Now let's create our memory instance - the brain of our research assistant:

In [ ]:
# No try/except here: if the configuration is wrong, the cell should fail loudly.
memory = Memory.from_config(config)
print("Memory system initialized successfully!")

## Building the Research Assistant

Now we'll build our Personal Research Assistant using Mem0's vector memory capabilities.

### Core Mem0 Operations

Before we build the assistant, let's understand Mem0's main operations:

`memory.add()` - Stores new information
- Automatically extracts key facts from conversations
- ADD-only in mem0 2.x: `add()` only adds new facts, it no longer rewrites or deletes existing memories
- Stores memories with user-specific isolation (`add()` takes `user_id=...` directly)

![mem0_add_architecture](Assets/mem0_add_architecture.jpg)

`memory.search()` - Retrieves relevant memories  
- Finds semantically similar content, plus BM25 keyword overlap and shared entities when the `[nlp]` extra and `fastembed` are installed
- Returns ranked results by relevance
- Scoped with `filters={"user_id": ...}`: mem0 2.x raises `ValueError` if `search()` or `get_all()` gets `user_id=` directly

![mem0_search_architecture](Assets/mem0_search_architecture.jpg)

*Like the overview above, these two diagrams show the mem0 1.x design, with an update step and a graph database. In mem0 2.x writes are ADD-only and the graph database is replaced by entity matching (see Phase 2).*

**Other operations** (not used in this tutorial):
- `memory.update()` - Modify existing memories
- `memory.delete()` - Remove specific memories

In [ ]:
class PersonalResearchAssistant:

    def __init__(self, memory_instance):
        self.client = OpenAI()
        self.memory = memory_instance  # Use the configured memory instance
        print(f"Research Assistant initialized with Mem0 memory!")

    def ask(self, question, user_id):

        # Search for relevant memories
        previous_memories = self.search_memories(question, user_id=user_id)

        # Build the prompt with memory context
        system_message = "You are a personal AI Research Assistant. Help users with research questions, remember their interests, and provide contextual recommendations."

        if previous_memories:
            memory_context = ", ".join(previous_memories)
            prompt = f"{system_message}\n\nUser input: {question}\nPrevious memories: {memory_context}"
        else:
            prompt = f"{system_message}\n\nUser input: {question}"

        # Generate response. Errors are not caught: a failed call should stop the cell, not come back as an "answer".
        response = self.client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": prompt},
            ],
            temperature=0.1,
            reasoning_effort="none",  # GPT-5.x models honor a custom temperature only with reasoning off
            max_completion_tokens=2000
        )

        answer = response.choices[0].message.content

        # Store the question in memory (add() takes user_id directly)
        self.memory.add(question, user_id=user_id, metadata={"category": "research"})

        return answer

    # mem0 2.x: get_all() and search() take the user inside `filters`.
    # Passing user_id=... directly raises ValueError, so we let errors surface instead of returning [].
    def get_memories(self, user_id):
        memories = self.memory.get_all(filters={"user_id": user_id})
        return [m["memory"] for m in memories["results"]]

    def search_memories(self, query, user_id):
        memories = self.memory.search(query, filters={"user_id": user_id})
        return [m["memory"] for m in memories["results"]]

# Initialize our research assistant
assistant = PersonalResearchAssistant(memory)
print("\n Research Assistant ready to learn and help!")

### Test 1: First Interaction - Knowledge Extraction

Let's start with a basic research interest:

In [ ]:
response1 = assistant.ask(
    "I'm interested in transformer architectures for natural language processing. "
    "Can you help me find recent papers on this topic?",
    user_id="researcher"
)

print(f"Assistant: {response1}")

### Test 2: Building Context - Watch Memory Connect Ideas

In [ ]:
response2 = assistant.ask(
    "I prefer papers that include practical implementation details and code examples. "
    "Theoretical papers without code are less useful for my work.",
    user_id="researcher"
)

print(f"Assistant: {response2}")

### Test 3: Semantic Search - Intelligence Beyond Keywords

In [ ]:
response3 = assistant.ask(
    "What about BERT and GPT models? Are they related to my research interests?",
    user_id="researcher"
)

print(f"Assistant: {response3}")

### Test 4: A Changed Preference - What ADD-only Memory Does

In Test 2 you said theoretical papers without code are less useful. Now you ask for theory papers. mem0 1.x ran an update step that could rewrite or delete the older memory. mem0 2.x is ADD-only: it stores the new fact next to the old one and leaves it to search to surface whichever is relevant to the next question.

In [ ]:
response4 = assistant.ask(
    "Actually, I also need to understand the theoretical foundations of attention mechanisms. "
    "Can you recommend some foundational theory papers?",
    user_id="researcher"
)

print(f"Assistant: {response4}")

## 📊 Memory Analysis - See What Was Learned

Let's examine what our assistant learned from these interactions. Because writes are ADD-only, expect to see the old preference and the new one side by side, and sometimes two near-duplicate phrasings of the same fact:

In [ ]:
def analyze_extracted_memories():

    print("📈 MEMORY ANALYSIS - What Did the Assistant Learn?")
    print("=" * 60)

    # Get all memories
    all_memories = assistant.get_memories(user_id="researcher")

    if all_memories:
        print(f"\n🧠 Total memories extracted: {len(all_memories)}")
        print(f"\n📚 Key insights Mem0 learned about you:")

        for i, memory in enumerate(all_memories, 1):
            print(f"\n{i}. {memory}")

        # Test semantic search
        print(f"\n🔍 Testing Semantic Search Capabilities:")

        test_queries = [
            "neural networks",           # Should connect to transformers
            "code implementations",      # Should find practical preferences
            "attention mechanisms",      # Should connect to transformer interest
            "deep learning papers"       # Should find research interests
        ]

        for query in test_queries:
            related_memories = assistant.search_memories(query, user_id="researcher")
            print(f"\n   🔎 Query: '{query}'")
            print(f"      Found {len(related_memories)} related memories")
            if related_memories:
                print(f"      Top match: {related_memories[0][:100]}...")

    else:
        print("\n⚠️ No memories found. Try running the interaction tests first.")

    return all_memories

# Run the analysis
memories = analyze_extracted_memories()

## Interactive Demo - Try It Yourself

Now it's your turn! Try asking questions and watch the memory system learn about your research interests:

In [ ]:
def interactive_demo():
    while True:
        try:
            question = input("\nYou: ")

            if question.lower() in ['quit', 'exit', 'q']:
                print("\n Thanks for trying the Research Assistant!")
                break
            elif question.lower() == 'memories':
                memories = assistant.get_memories(user_id="researcher")
                if memories:
                    print(f"\n Here's what I've learned about your research interests:")
                    for i, memory in enumerate(memories, 1):
                        print(f"   {i}. {memory}")
                else:
                    print("\n No memories yet. Start asking about your research interests!")
            elif question.strip():
                response = assistant.ask(question, user_id="researcher")
                print(f"\n Assistant: {response}")

        except (KeyboardInterrupt, EOFError):
            print("\n\n Demo ended. Thanks for trying the Research Assistant!")
            break

# Run the interactive demo
interactive_demo()

## 🚀 Phase 2: Entity-Aware Retrieval

Phase 1 retrieved memories by meaning. Research questions are also full of names: papers, authors, models. A query like "papers by Vaswani" should find the memory that mentions Vaswani, even when the rest of the wording is different.

Earlier versions of this tutorial added a Neo4j graph store for this. **mem0 2.0 removed external graph stores from the open-source package**, and graph memory now exists only in the hosted Mem0 Platform. If you need the Neo4j version, pin `mem0ai<2`.

What open-source mem0 2.x does instead:

- **Entity extraction on write**: when `add()` stores a memory, spaCy pulls out the entities it mentions (people, paper titles, models) and keeps them in a small entity index next to the vectors, linked to that memory.
- **Entity boost on read**: `search()` extracts the entities in the query, and memories linked to the same entities get a boost on top of their semantic and BM25 keyword scores.

### What entity matching is not

It is not a graph. It boosts memories that mention the same names, but it stores no typed relationships (who cited whom, who worked with whom) and cannot traverse them. Questions like "How did Paper A influence Paper B?" still need a graph database you run yourself, or a hosted graph memory.

### Check that entity extraction is on

Entity extraction needs the `[nlp]` extra and the spaCy English model from the install step, and BM25 needs `fastembed`. Without them Mem0 quietly falls back to semantic-only search, so it is worth checking before you rely on it:

In [ ]:
import importlib.util
import spacy

assert importlib.util.find_spec("fastembed"), "fastembed is missing, so BM25 keyword search is off. Run: pip install fastembed"
spacy.load("en_core_web_sm")  # raises OSError if the model is missing: python -m spacy download en_core_web_sm
print("Entity extraction (spaCy) and BM25 keyword search (fastembed) are available.")

### 📝 Entity-Aware Research Examples

No new configuration is needed: entity extraction already runs inside the `memory` from Phase 1. We use a separate `user_id` so these examples don't mix with the first researcher's memories. Both questions are full of names:

### Example 1: Research lineage mapping

In [ ]:
entity_response1 = assistant.ask(
    "I'm studying the lineage of transformer papers. The original 'Attention Is All You Need' "
    "by Vaswani et al. led to BERT by Devlin et al., and then to many other models. "
    "Can you help me map these research connections and suggest related work?",
    user_id="lineage_user"
)

print(f"Assistant: {entity_response1}")

### Example 2: Collaboration network analysis

In [ ]:
entity_response2 = assistant.ask(
    "What other researchers have worked on transformer architectures? "
    "I want to understand the collaboration network and research groups in this field.",
    user_id="lineage_user"
)

print(f"Assistant: {entity_response2}")

### See the score breakdown

`search(..., explain=True)` returns `score_details` for every hit: the semantic score, the BM25 keyword score and the entity boost. Let's look at two queries that name an entity and one that doesn't:

In [ ]:
def explain_search(query, user_id="lineage_user", top_k=3):
    results = memory.search(query, filters={"user_id": user_id}, top_k=top_k, explain=True)
    print(f"\n🔎 {query}")
    for hit in results["results"]:
        d = hit["score_details"]
        print(f"   final={d['final_score']:.2f}  semantic={d['semantic_score']:.2f}  "
              f"bm25={d['bm25_score']:.2f}  entity={d['entity_boost']:.2f}  | {hit['memory'][:80]}")

for query in ["papers by Vaswani", "Who wrote BERT?", "how do research groups collaborate?"]:
    explain_search(query)

**Reading the numbers.** A memory that mentions Vaswani or BERT gets an entity boost (about 0.5 in our runs) on top of its semantic and BM25 scores, while a memory that mentions neither gets 0 and drops down the list. Which memory mentions which name depends on what the LLM extracted, so your exact scores and order will differ from run to run.

The third query names no paper or person, yet the memory about research groups still got an entity boost: Mem0 extracts topic phrases such as "research groups" as entities too, not only proper names. That is useful, and it is also the limit of this approach: entities are matched as strings, so the memory never learns that Devlin's BERT *built on* Vaswani's Transformer. For relationships you can query, you need a graph.

## 🎓 Recap

You just built a real memory system from the ground up. Here's what you accomplished and where to go from here.

### What you built
- **Hybrid retrieval** that combines vector search (Qdrant), BM25 keyword matching and entity matching
- **Automatic memory extraction** that pulls key facts from conversations  
- **ADD-only memory** that keeps every fact it extracts, old and new, and lets search decide what is relevant
- **Semantic search** that finds relevant info by meaning, not just keywords

### Why Mem0

Memory is hard to build right. You'd need months to handle fact extraction, conflict resolution, and semantic search properly. Mem0 gives you battle-tested memory that just works.

**What you get:**
- Smart memory extraction that pulls key facts from conversations automatically
- Changed preferences kept side by side, with search surfacing the one that fits the question  
- Semantic search that finds relevant stuff by meaning, not just keywords
- Proven performance: 26% better accuracy, 91% faster responses, 90% fewer tokens (Mem0 research paper)

You can focus on building your app instead of reinventing memory infrastructure.

### Your deployment options

**Self-hosted (what we built)**
- Full control over your stack  
- Custom configurations for vector stores
- Great for learning, prototyping, and specific requirements

**Mem0 Platform**
- Managed service starting at free tier (10K memories)
- Enterprise features like analytics and graph memory  
- Simple API, zero infrastructure headaches
- Ship fast and scale without ops overhead


### What's next
You now have the foundation for AI that actually remembers and learns. Check out:

- **[Mem0 Platform](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdashboard%2Fnir&text=Mem0%20Platform)** for production deployment
- **[Documentation](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdocs%2Fnir&text=Documentation)** for advanced features and integrations
- **[Discord Community](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdiscord%2Fnir&text=Discord%20Community)** for questions and discussions

You're ready to build AI applications that get smarter with every interaction.

---
📖 **Want the decisions behind agents like this one, explained without code?** [*AI Agents Made Simple*](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=book-nb-agents&target=https%3A%2F%2Fwww.amazon.com%2Fdp%2FB0HNXRSRH2&retarget=0&text=book-nb-agents) by the author of this repository covers this and much more. Available on Amazon.